# Etapa 2 — Screening de Selección (`selection_campaign_1200_50`) y Poda de Modelos

Replica `tools/selection_campaign_1200_50.py` e implementa la lógica de **screening cuantitativo en dos pasadas** con **detección temprana de divergencia**:
1. **Pasada 1 (`with_energy`, `"energy": 100.0`)** y **Pasada 2 (`no_energy`, `"energy": 0.0`)**: entrena todas las arquitecturas candidatas con presupuesto intermedio (`4000 Adam + 200 L-BFGS` en `full_campaign`; escala rápida en `demo_fast`) evaluando frente a referencia analítica y **FEM WSL2**. Si una corrida diverge (`NaN`/`Inf`, explosión o crecimiento sostenido de pérdida), `divergence_stop` la detiene inmediatamente sin esperar a completar todos los pasos.
2. Calcula el puntaje compuesto adimensional (error en $T_{\max}$, `NRMSE`, desbalance energético y robustez en cable frente a `mms_sine`) sobre **ambas pasadas** para seleccionar el mejor modo energético de cada arquitectura y filtrar modelos que se estancan o divergen.
3. Escribe la lista curada de **modelos promovidos** en `campaign_workspace/promoted_models.json`, reteniendo explícitamente si cada modelo clasificó **con `energy`** (`100.0`) o **sin `energy`** (`0.0`) en la función de pérdida.

In [ ]:
import json
import os
import sys
import time
from pathlib import Path
import numpy as np

# Localiza la raíz PINN_V4 aunque el cuaderno se ejecute dentro de notebooks_campaign/
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks_campaign":
    ROOT = ROOT.parent
SRC = ROOT / "src"
for p in (SRC, ROOT):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

# Modo de ejecución:
# - 'demo_fast': subconjunto acotado de casos y pocos pasos para verificación rápida en minutos
# - 'full_campaign': reproduce todos los casos, semillas (0,1,2) y presupuestos largos de la tesis
RUN_MODE = "full_campaign"    #"demo_fast"

# Paralelismo de campaña:
# - USE_PARALLEL=True  -> fuerza device='cpu' con ParallelRunner multi-proceso
# - USE_PARALLEL=False -> ejecuta secuencialmente en device='auto' (p. ej. XPU nativo sin fallbacks)
USE_PARALLEL = True
MAX_WORKERS = min(12, os.cpu_count() or 1)
DEVICE = "cpu" if USE_PARALLEL else "auto"

CAMP_WS = ROOT / "notebooks_campaign" / "campaign_workspace"
FEM_CACHE = CAMP_WS / "fem_references"
CAMP_WS.mkdir(parents=True, exist_ok=True)
FEM_CACHE.mkdir(parents=True, exist_ok=True)

# Criterios oficiales de calificación de la campaña final de tesis (2026-10-27)
STRICT_CRITERIA = {"error_Tmax_abs_K": 0.05, "nrmse_pct": 0.25, "balance_pct": 0.5}
RELAXED_CRITERIA = {k: 2.0 * v for k, v in STRICT_CRITERIA.items()}

print(f"PINN_V4 raíz: {ROOT} | RUN_MODE={RUN_MODE!r} | USE_PARALLEL={USE_PARALLEL} (workers={MAX_WORKERS}, device={DEVICE!r}) | Workspace campaña: {CAMP_WS}")

In [ ]:
import pinn_thermal.models  # noqa: F401
from pinn_thermal.core.artifact import SolutionArtifact
from pinn_thermal.core.manifest import write_json_atomic
from pinn_thermal.core.parallel import ParallelRunner, RunTask
from pinn_thermal.core.problem import CaseCatalog
from pinn_thermal.core.trainer import PINNTrainer, TrainConfig
from pinn_thermal.fem.runner import FEniCSxAdapter

catalog = CaseCatalog(ROOT / "cases")

def load_or_solve_reference(case_id: str, problem, use_fem: bool = True):
    """Obtiene referencia analítica exacta si existe, o carga/resuelve caché FEM P2 en WSL."""
    xy, _, _ = problem.evaluation()
    exact = problem.reference(xy)
    if exact is not None and np.isfinite(exact).all():
        return np.asarray(exact, dtype=float), "analytic_reference"

    fem_dir = FEM_CACHE / case_id
    fem_json = fem_dir / "fem_l1.json"
    if fem_json.is_file() and fem_json.with_suffix(".npz").is_file():
        art = SolutionArtifact.load(fem_json)
        return art.predict(xy)["T"], f"FEM_WSL_cache:{case_id}"

    if use_fem and FEniCSxAdapter().is_available().get("available"):
        art = FEniCSxAdapter().solve(problem, output_dir=fem_dir, level=1)
        return art.predict(xy)["T"], f"FEM_WSL_L1:{case_id}"

    return None, "physics_only_no_reference"

def extract_metrics(result: dict) -> dict:
    ev = result.get("evaluation") or {}
    bal = ev.get("balance") or {}
    return {
        "error_Tmax_abs_K": ev.get("error_Tmax_abs_K"),
        "nrmse_pct": ev.get("nrmse_pct"),
        "balance_pct": bal.get("global_derived_pct"),
        "Tmax_pred_C": ev.get("Tmax_pred_C"),
        "elapsed_s": result.get("elapsed_s"),
    }
CASES_STAGE2 = ("mms_sine", "cable_disk")
CANDIDATE_MODELS_S2 = (
    ("M0", "M1", "M1_EXACT", "CUSPR", "M0R", "M2")
    if RUN_MODE == "demo_fast"
    else ("M0", "M1", "M2", "DCSNN", "M0R", "CUSP", "CUSPR", "FBPINN", "M1_EXACT", "CUSP_EXACT")
)
ADAM_S2 = 400 if RUN_MODE == "demo_fast" else 4000
LBFGS_S2 = 15 if RUN_MODE == "demo_fast" else 200

ENERGY_PASSES_S2 = (
    {"label": "with_energy", "use_energy_loss": True, "energy_weight": 100.0},
    {"label": "no_energy", "use_energy_loss": False, "energy_weight": 0.0},
)
DIVERGENCE_STOP_S2 = {
    "max_loss": 1e12,
    "explosion_factor": 1e4,
    "patience_explosion": 3,
    "growth_patience": 6,
    "min_growth_ratio": 1.05,
    "warmup_steps": 150,
    "lbfgs_explosion_factor": 100.0,
    "lbfgs_stagnation_checks": 8,
    "check_nonfinite_grads": True,
}

stage2_dir = CAMP_WS / "stage2_selection_4000_200"
stage2_dir.mkdir(parents=True, exist_ok=True)
stage2_summary_path = CAMP_WS / "stage2_selection_summary.json"

def run_stage2_pass(pass_cfg: dict) -> dict:
    """Ejecuta una pasada completa de screening (con o sin término energy) con parada temprana por divergencia."""
    label = str(pass_cfg["label"])
    use_energy = bool(pass_cfg["use_energy_loss"])
    energy_weight = float(pass_cfg["energy_weight"])
    summary = json.loads(stage2_summary_path.read_text(encoding="utf-8-sig")) if stage2_summary_path.is_file() else {}
    tasks = []
    for case_id in CASES_STAGE2:
        prob = catalog.load(case_id)
        ref_vals, ref_lbl = load_or_solve_reference(case_id, prob, use_fem=True)
        for arch in CANDIDATE_MODELS_S2:
            key = f"{case_id}:{arch}:{label}"
            run_dir = stage2_dir / label / case_id / arch
            tasks.append(
                RunTask(
                    name=key,
                    problem_source=prob.doc,
                    architecture=arch,
                    output_dir=run_dir,
                    config_kwargs={
                        "adam_steps": ADAM_S2,
                        "lbfgs_steps": LBFGS_S2,
                        "lr": 2e-3,
                        "width": 32,
                        "depth": 3,
                        "weights": {"pde": 1.0, "energy": energy_weight},
                        "exact_energy": use_energy,
                        "sampling": "mixed",
                        "focused": True,
                        "focus_criterion": "residual",
                        "focus_surfaces": False,
                        "resample_every": 100,
                        "full_audit": True,
                        "divergence_stop": DIVERGENCE_STOP_S2,
                        "device": DEVICE,
                        "dtype": "float32",
                    },
                    reference=ref_vals,
                    reference_label=ref_lbl,
                    overwrite=True,
                )
            )

    def _on_done(res: dict) -> None:
        key = res["name"]
        case_id, arch, _ = key.split(":", 2)
        m = extract_metrics(res)
        rec = {
            "case": case_id,
            "architecture": arch,
            "energy_mode": label,
            "use_energy_loss": use_energy,
            "energy_weight": energy_weight,
            "exact_energy": use_energy,
            "status": res.get("status"),
            "divergence_stop": res.get("divergence_stop"),
            **m,
        }
        if res.get("error"):
            rec["error"] = res.get("error")
        summary[key] = rec
        write_json_atomic(stage2_summary_path, summary)
        if rec["status"] == "diverged":
            div = rec.get("divergence_stop") or {}
            print(f"[Etapa 2 | {label}] {key:32s} | DIVERGED @ step={div.get('step')} ({div.get('reason')})")
        elif rec["status"] != "completed":
            print(f"[Etapa 2 | {label}] {key:32s} | FAILED: {rec.get('error')}")
        else:
            print(f"[Etapa 2 | {label}] {key:32s} | dev={res.get('device')} | NRMSE={m['nrmse_pct']} | dTmax={m['error_Tmax_abs_K']} K | bal={m['balance_pct']}%")

    runner = ParallelRunner(max_workers=MAX_WORKERS if USE_PARALLEL else 1, distribute_devices=not USE_PARALLEL)
    runner.run(tasks, on_complete=_on_done, show_progress=True, progress_title=f"Etapa 2 — {label} (energy={energy_weight:g})")
    write_json_atomic(stage2_summary_path, summary)
    return summary

In [ ]:
# Pasada 1: Screening con término energético ("energy": 100.0)
stage2_summary = run_stage2_pass(ENERGY_PASSES_S2[0])

## Pasada 2 — Screening sin término energético (`"energy": 0.0`)

In [ ]:
# Pasada 2: Screening sin término energético ("energy": 0.0)
stage2_summary = run_stage2_pass(ENERGY_PASSES_S2[1])

## Lógica de Screening y Promoción de Modelos

In [ ]:
# Agregación multi-caso y multi-pasada (with_energy vs no_energy) + regla de poda
if stage2_summary_path.is_file():
    stage2_summary = json.loads(stage2_summary_path.read_text(encoding="utf-8-sig"))

per_variant = {}
diverged_runs = []
for key, rec in stage2_summary.items():
    energy_mode = rec.get("energy_mode") or "with_energy"
    use_energy = bool(rec["use_energy_loss"]) if "use_energy_loss" in rec else (energy_mode == "with_energy")
    energy_weight = float(rec["energy_weight"]) if "energy_weight" in rec else (100.0 if use_energy else 0.0)
    exact_energy = bool(rec["exact_energy"]) if "exact_energy" in rec else use_energy

    if rec.get("status") == "diverged":
        div = rec.get("divergence_stop") or {}
        diverged_runs.append({
            "key": key,
            "case": rec.get("case"),
            "architecture": rec.get("architecture"),
            "energy_mode": energy_mode,
            "energy_weight": energy_weight,
            "step": div.get("step"),
            "phase": div.get("phase"),
            "reason": div.get("reason"),
        })
        continue
    if rec.get("status") != "completed":
        continue

    arch = rec["architecture"]
    vkey = (arch, energy_mode)
    row = per_variant.setdefault(
        vkey,
        {
            "architecture": arch,
            "energy_mode": energy_mode,
            "use_energy_loss": use_energy,
            "energy_weight": energy_weight,
            "exact_energy": exact_energy,
            "scores": [],
            "nrmse": [],
            "balance": [],
            "dtmax": [],
            "strict_passed": 0,
            "elapsed": [],
        },
    )
    bal = rec["balance_pct"] if rec["balance_pct"] is not None else 100.0
    nrmse = rec["nrmse_pct"] if rec["nrmse_pct"] is not None else 100.0
    dtmax = rec["error_Tmax_abs_K"] if rec["error_Tmax_abs_K"] is not None else 100.0
    score = (dtmax / STRICT_CRITERIA["error_Tmax_abs_K"]) + (nrmse / STRICT_CRITERIA["nrmse_pct"]) + (bal / STRICT_CRITERIA["balance_pct"])
    row["scores"].append(score)
    row["nrmse"].append(nrmse)
    row["balance"].append(bal)
    row["dtmax"].append(dtmax)
    if dtmax < STRICT_CRITERIA["error_Tmax_abs_K"] and nrmse < STRICT_CRITERIA["nrmse_pct"] and bal < STRICT_CRITERIA["balance_pct"]:
        row["strict_passed"] += 1
    row["elapsed"].append(rec.get("elapsed_s") or 0.0)

ranking = []
for (_arch, _mode), vals in per_variant.items():
    if len(vals["scores"]) == len(CASES_STAGE2):
        ranking.append({
            "architecture": vals["architecture"],
            "energy_mode": vals["energy_mode"],
            "use_energy_loss": vals["use_energy_loss"],
            "energy_weight": vals["energy_weight"],
            "exact_energy": vals["exact_energy"],
            "mean_score": float(np.mean(vals["scores"])),
            "max_balance_pct": float(np.max(vals["balance"])),
            "mean_nrmse_pct": float(np.mean(vals["nrmse"])),
            "mean_error_Tmax_abs_K": float(np.mean(vals["dtmax"])),
            "strict_cases_passed": int(vals["strict_passed"]),
            "total_elapsed_s": float(np.sum(vals["elapsed"])),
        })
ranking.sort(key=lambda r: r["mean_score"])

# Mejor configuración (con energy o sin energy) por arquitectura evaluando ambas pasadas
THESIS_RETAINED = ["M1", "M1_EXACT", "CUSPR", "M0R", "M2"]
by_arch = {}
for item in ranking:
    by_arch.setdefault(item["architecture"], []).append(item)

best_per_architecture = []
for arch, variants in by_arch.items():
    variants_sorted = sorted(variants, key=lambda r: r["mean_score"])
    winner = dict(variants_sorted[0])
    winner["in_thesis_retained"] = arch in THESIS_RETAINED
    if len(variants_sorted) > 1:
        other = variants_sorted[1]
        winner["comparison_other_pass"] = {
            "energy_mode": other["energy_mode"],
            "energy_weight": other["energy_weight"],
            "mean_score": other["mean_score"],
            "mean_nrmse_pct": other["mean_nrmse_pct"],
            "max_balance_pct": other["max_balance_pct"],
        }
    else:
        other_divs = [d for d in diverged_runs if d["architecture"] == arch and d["energy_mode"] != winner["energy_mode"]]
        winner["comparison_other_pass"] = {"status": "diverged", "runs": other_divs} if other_divs else None
    best_per_architecture.append(winner)
best_per_architecture.sort(key=lambda r: r["mean_score"])

promoted_models_detail = best_per_architecture[:5]
promoted_models = [r["architecture"] for r in promoted_models_detail]
long_run_finalists_detail = promoted_models_detail[:2]
finalists = [r["architecture"] for r in long_run_finalists_detail]
promoted_energy_config = {
    r["architecture"]: {
        "energy_mode": r["energy_mode"],
        "use_energy_loss": r["use_energy_loss"],
        "energy_weight": r["energy_weight"],
        "exact_energy": r["exact_energy"],
    }
    for r in promoted_models_detail
}

payload = {
    "run_mode": RUN_MODE,
    "energy_passes": list(ENERGY_PASSES_S2),
    "divergence_stop": DIVERGENCE_STOP_S2,
    "ranking": ranking,
    "best_per_architecture": best_per_architecture,
    "diverged_runs": diverged_runs,
    "promoted_models": promoted_models,
    "promoted_models_detail": promoted_models_detail,
    "promoted_energy_config": promoted_energy_config,
    "long_run_finalists": finalists,
    "long_run_finalists_detail": long_run_finalists_detail,
}
write_json_atomic(CAMP_WS / "promoted_models.json", payload)

print("=== Modelos promovidos (considerando ambas pasadas: con/sin energy) ===")
for r in promoted_models_detail:
    tag = "CON energy (100.0)" if r["use_energy_loss"] else "SIN energy (0.0)"
    print(
        f"- {r['architecture']:10s} | clasificó: {tag:18s} | score={r['mean_score']:8.2f} "
        f"| NRMSE={r['mean_nrmse_pct']:.4f}% | max_bal={r['max_balance_pct']:.4f}% | thesis={r['in_thesis_retained']}"
    )
if diverged_runs:
    print(f"\nCorridas detenidas por divergencia ({len(diverged_runs)}):", json.dumps(diverged_runs, indent=2))
print("\nModelos promovidos a Etapa 3/4:", promoted_models)
print("Finalistas corridas largas:", finalists)